## Мультиагентная система по образу общества Лумана для моделирования динамики реакции функциональной коммуникативной системы на шок

### Нет гейммастера. Результат д.б. в виде набора триплетов: (кто, что сделал, с чем)
смотрим набор мер от агентов разворачивающийся во времени

In [2]:

pip show langchain langchain-openai

Name: langchain
Version: 1.3.18
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: c:\Users\user\Documents\PythonScripts\LLM_CSS\Luhmann\MAS_reaction_on_shock\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
import langchain
import langchain_core
import langchain_openai
import langgraph

print("Python:", sys.executable)
print("langchain:", langchain.__version__)
print("langchain_core:", langchain_core.__version__)
print("langchain_openai:", langchain_openai.__version__)
print("langgraph:", langgraph.__version__)

ModuleNotFoundError: No module named 'langchain_openai'

In [1]:
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

import os
import json
import time
import operator
import re

from typing import List, Dict, Any, TypedDict, Annotated, Optional
from langgraph.graph import StateGraph, END

from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type, RetryError
from openai import InternalServerError, APIError, APIConnectionError, Timeout
from httpx import RemoteProtocolError, ReadTimeout, ConnectError

API_KEY = os.getenv("OPENAI_API_KEY")
URL = "https://foundation-models.api.cloud.ru/v1"
#BASE_MODEL = "Qwen/Qwen3-235B-A22B-Instruct-2507"
BASE_MODEL = "openai/gpt-5.1" # 200 руб за 10 периодов
#BASE_MODEL = "openai/gpt-5" # 200 руб за 10 периодов
#BASE_MODEL = "openai/gpt-5.4-nano" # тупенькая

#RES_DIR = "c:/Users/PC1/Documents/PythonScripts/LLM CSS/Luhmann/MAS_reaction_on_shock/Covid_experiments/ru/ag_v2/2020/РФ2020_"
RES_DIR = "c:/Users/PC1/Documents/PythonScripts/LLM CSS/Luhmann/MAS_reaction_on_shock/Covid_experiments/ru/ag_v2/2000_1/РФ2000_gpt51_"
timestamp = time.strftime('%Y%m%d-%H%M')
RES_DIR = RES_DIR + timestamp
PROMPT_DIR = "c:/Users/PC1/Documents/PythonScripts/LLM CSS/Luhmann/MAS_reaction_on_shock/Prompts/ru/ag_v2/2000_1"



#TEMPERATURE = 0.5
TEMPERATURE = 1

TARGET_YEAR = 10
RECURSION_LIMIT = max(100, TARGET_YEAR * 25)

INPUT_MSG = """Минздрав сообщил о первых случаях распространения нового штамма вируса внутри страны.
Заболевание может протекать как в форме острой респираторной вирусной инфекции лёгкого течения, так и в тяжёлой форме. 
К наиболее распространённым симптомам заболевания относятся повышенная температура тела, утомляемость и сухой кашель.
Распространяется воздушно-капельным путём через вдыхание распылённых в воздухе при кашле, чихании или разговоре капель с вирусом, 
а также через попадание вируса на поверхности с последующим занесением в глаза, нос или рот."""

os.makedirs(RES_DIR, exist_ok=True)

def save_experimet_info(prompt_dir, input_msg, temp, ex_time, model):
    """
    Сохраняет параметры эксперимента в отдельный файл.
    """    
    filename = f"info_{timestamp}.txt"
    metadata = {
        "prompt_dir": prompt_dir,
        "input_msg": input_msg,
        "temperature": temp,
        "execution time": ex_time,
        "base_model": model,
    }
    filepath = os.path.join(RES_DIR, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    print(f"Сохранено: {filepath}")
    return filepath

class Message(TypedDict):
    sender: str
    sender_org: str
    recipient: str
    recipient_org: str
    action: str
    year: int
    metadata: Dict[str, Any]

def replace_year(current: int, new: int) -> int:
    return new


class AgentState(TypedDict):
    year: Annotated[int, replace_year]
    target_year: int

    message_history: Annotated[List[Message], operator.add] # json in mem. str for LLM

    current_year_messages: List[Message]
    next_year_messages: List[Message]

    agents_state: Dict[str, Any]

    user_input: str
    output: Optional[str]
    step_count: int


def create_agent_executor(system_prompt: str, tools: list):
    llm = ChatOpenAI(
        model=BASE_MODEL,
        api_key=API_KEY,
        base_url=URL,
        temperature=TEMPERATURE
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}")
    ])

    agent = create_openai_tools_agent(llm, tools, prompt)

    return AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=True,
        return_intermediate_steps=True
    )

with open(PROMPT_DIR+'/Системный промпт политика.txt', encoding='utf-8') as f:
    POLITICS_PROMPT = f.read()
#print(POLITICS_PROMPT)

with open(PROMPT_DIR+'/Системный промпт право.txt', encoding='utf-8') as f:
    LAW_PROMPT = f.read()

with open(PROMPT_DIR+'/Системный промпт наука.txt', encoding='utf-8') as f:
    SCIENCE_PROMPT = f.read()

with open(PROMPT_DIR+'/Системный промпт экономика.txt', encoding='utf-8') as f:
    ECONOMICS_PROMPT = f.read()

with open(PROMPT_DIR+'/Системный промпт медиа.txt', encoding='utf-8') as f:
    MEDIA_PROMPT = f.read()


# =========================
# TOOLS
# =========================

@tool
def send_to_politics(message: str):
    """Отправить структурированное сообщение в подсистему politics.
    Сообщение должно быть в формате JSON с полями:
    sender_subsystem, sender_org, recipient_subsystem, recipient_org, action."""
    return f"TOOL_MESSAGE: politics|{message}"

@tool
def send_to_law(message: str):
    """Отправить структурированное сообщение в подсистему law.
    Сообщение должно быть в формате JSON с полями:
    sender_subsystem, sender_org, recipient_subsystem, recipient_org, action."""
    return f"TOOL_MESSAGE: law|{message}"

@tool
def send_to_science(message: str):
    """Отправить структурированное сообщение в подсистему science.
    Сообщение должно быть в формате JSON с полями:
    sender_subsystem, sender_org, recipient_subsystem, recipient_org, action."""
    return f"TOOL_MESSAGE: science|{message}"

@tool
def send_to_media(message: str):
    """Отправить структурированное сообщение в подсистему media.
    Сообщение должно быть в формате JSON с полями:
    sender_subsystem, sender_org, recipient_subsystem, recipient_org, action."""
    return f"TOOL_MESSAGE: media|{message}"

@tool
def send_to_economics(message: str):
    """Отправить структурированное сообщение в подсистему economics.
    Сообщение должно быть в формате JSON с полями:
    sender_subsystem, sender_org, recipient_subsystem, recipient_org, action."""
    return f"TOOL_MESSAGE: economics|{message}"


# =========================
# НАБОРЫ ИНСТРУМЕНТОВ
# =========================

law_tools = [send_to_politics, send_to_economics, send_to_science, send_to_media]
politics_tools = [send_to_law, send_to_economics, send_to_science, send_to_media]
science_tools = [send_to_law, send_to_economics, send_to_media, send_to_politics]
economics_tools = [send_to_law, send_to_science, send_to_media, send_to_politics]
media_tools = [send_to_law, send_to_economics, send_to_science, send_to_politics]


# =========================
# СОЗДАНИЕ АГЕНТОВ
# =========================

law_agent = create_agent_executor(LAW_PROMPT, law_tools)
politics_agent = create_agent_executor(POLITICS_PROMPT, politics_tools)
science_agent = create_agent_executor(SCIENCE_PROMPT, science_tools)
economics_agent = create_agent_executor(ECONOMICS_PROMPT, economics_tools)
media_agent = create_agent_executor(MEDIA_PROMPT, media_tools)

# =========================
# HISTORY → CHAT
# =========================

def format_history(messages: List[Message]) -> List[Dict[str, str]]:
    formatted = []

    for msg in messages[-20:]:
        formatted.append({
            "role": "system",
            "content": (
                f"[Год {msg['year']}] "
                f"{msg['sender']}/{msg['sender_org']} → "
                f"{msg['recipient']}/{msg['recipient_org']}: "
                f"{msg['action']}"
            )
        })

    return formatted

# =========================
# PARSER
# =========================

def extract_messages_from_output(output: str, year: int) -> List[Message]:
    pattern = r"TOOL_MESSAGE:\s*(\w+)\|(.*)"
    matches = re.findall(pattern, output.strip())

    messages: List[Message] = []

    for _, json_str in matches:
        if json_str.count('{') > json_str.count('}'):
            json_str += '}'

        try:
            data = json.loads(json_str)

            msg: Message = {
                "sender": data["sender_subsystem"],
                "sender_org": data["sender_org"],
                "recipient": data["recipient_subsystem"],
                "recipient_org": data["recipient_org"],
                "action": data["action"],
                "year": year,
                "metadata": {}
            }

            messages.append(msg)

        except Exception as e:
            print(f"Ошибка парсинга: {e}")

    return messages


# =========================
# AGENT NODE
# =========================



def is_retryable_exception(exception):
    return isinstance(exception, (
        InternalServerError,
        RemoteProtocolError,
        ReadTimeout,
        ConnectError,
        APIConnectionError,
        Timeout
    ))

@retry(
    stop=stop_after_attempt(10),
    wait=wait_exponential(multiplier=1, min=5, max=15),
    retry=retry_if_exception_type((
    InternalServerError,
    RemoteProtocolError,
    ReadTimeout,
    ConnectError,
    APIConnectionError,
    Timeout
))
)
def invoke_agent_safe(input_text, filtered_history, agent_executor):
    result = agent_executor.invoke({
            "input": input_text,
            "chat_history": format_history(filtered_history) # state["message_history"]
        })
    return result

def make_agent_node(agent_name: str, agent_executor: AgentExecutor):

    def agent_node(state: AgentState) -> dict:

        my_messages = [m for m in state["current_year_messages"] if m["recipient"] == agent_name]
        other_messages = [m for m in state["current_year_messages"] if m["recipient"] != agent_name]

        input_text = f"Год {state['year']}.\n"

        if my_messages:
            input_text += "Поступили раздражения:\n"
            for msg in my_messages:
                input_text += f"- От {msg['sender']}/{msg['sender_org']}: {msg['action']}\n"
        else:
            input_text += "Новых раздражений не поступило."

        if state.get("user_input"):
            input_text += f"\nКонтекст: {state['user_input']}"

        filtered_history = [msg for msg in state["message_history"] if msg["recipient"] == agent_name or msg["sender"] == agent_name] # даем ЛЛМ только с ней связанные сообщения

        
        
        try:
            result = invoke_agent_safe(input_text, filtered_history, agent_executor)
        except RetryError as e:
            print(f"❌ Агент {agent_name} не ответил после 10 попыток: {e}")
            # Не добавляем сообщений, но продолжаем работу
            return {
                "current_year_messages": other_messages,
                "next_year_messages": state["next_year_messages"],
                "message_history": []
            }
        new_messages = []

        for _, obs in result.get("intermediate_steps", []):
            if isinstance(obs, str) and obs.startswith("TOOL_MESSAGE:"):
                new_messages.extend(extract_messages_from_output(obs, state["year"]))

        new_messages.extend(extract_messages_from_output(result["output"], state["year"]))

        return {
            "current_year_messages": other_messages,
            "next_year_messages": state["next_year_messages"] + new_messages,
            "message_history": new_messages
        }

    return agent_node


# =========================
# ROUTING
# =========================

def route_messages(state: AgentState) -> dict:
    return {"step_count": state.get("step_count", 0) + 1}


def route_decision(state: AgentState) -> str:
    if state.get("step_count", 0) > 20:
        return "finalize"

    if state["current_year_messages"]:
        return f"agent_{state['current_year_messages'][0]['recipient']}"

    return "finalize"


# =========================
# DISTRIBUTE
# =========================

def distribute_cycle(state: AgentState) -> dict:
    updates = {"step_count": 0, "output": None}

    if state["year"] == 1:
        start_messages = [
            {
                "sender": "system",
                "sender_org": "system",
                "recipient": agent,
                "recipient_org": "system",
                "action": "START_YEAR",
                "year": 1,
                "metadata": {}
            }
            for agent in ["law", "politics", "science", "economics", "media"]
        ]

        updates["current_year_messages"] = start_messages
        updates["message_history"] = start_messages

    return updates


# =========================
# FINALIZE
# =========================

def finalize(state: AgentState) -> dict:
    return {
        "year": state["year"] + 1,
        "current_year_messages": state["next_year_messages"],
        "next_year_messages": [],
        "output": f"Год завершён. Сообщений: {len(state['message_history'])}"
    }


def should_continue(state: AgentState) -> str:
    return "distribute" if state["year"] <= state["target_year"] else END


# =========================
# GRAPH
# =========================

graph = StateGraph(AgentState)

graph.add_node("distribute", distribute_cycle)
graph.add_node("route", route_messages)
graph.add_node("finalize", finalize)

graph.add_node("agent_law", make_agent_node("law", law_agent))
graph.add_node("agent_politics", make_agent_node("politics", politics_agent))
graph.add_node("agent_science", make_agent_node("science", science_agent))
graph.add_node("agent_economics", make_agent_node("economics", economics_agent))
graph.add_node("agent_media", make_agent_node("media", media_agent))

graph.set_entry_point("distribute")

graph.add_edge("distribute", "route")
graph.add_conditional_edges("route", route_decision)

for a in ["agent_law", "agent_politics", "agent_science", "agent_economics", "agent_media"]:
    graph.add_edge(a, "route")

graph.add_conditional_edges("finalize", should_continue)

app = graph.compile()


# =========================
# SAVE
# =========================

def save_full_history(messages: List[Message], filename):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(messages, f, ensure_ascii=False, indent=2)


# =========================
# RUN
# =========================

initial_state: AgentState = {
    "year": 1,
    "target_year": TARGET_YEAR,
    "user_input": INPUT_MSG,
    "agents_state": {},
    "current_year_messages": [],
    "next_year_messages": [],
    "message_history": [],
    "output": None,
    "step_count": 0
}

start = time.time()

for state in app.stream(initial_state, config={"recursion_limit": RECURSION_LIMIT}, stream_mode="values"):

    if state.get("output") is not None:
        year_completed = state["year"] - 1
        filename = f"{RES_DIR}/history_year_{year_completed}.json"
        save_full_history(state["message_history"], filename)

print("Финальный вывод:", state.get("output"))
execution_in = time.time()-start
print(f"Время: {(execution_in)/60:.2f} мин")

save_experimet_info(PROMPT_DIR, INPUT_MSG, TEMPERATURE, execution_in, BASE_MODEL)

ImportError: cannot import name 'create_openai_tools_agent' from 'langchain.agents' (c:\Users\user\Documents\PythonScripts\LLM_CSS\Luhmann\MAS_reaction_on_shock\Lib\site-packages\langchain\agents\__init__.py)

In [9]:
RES_DIR = "c:/Users/PC1/Documents/PythonScripts/LLM CSS/Luhmann/Covid_experiments/РФ2000_4"
os.makedirs(os.path.dirname(RES_DIR), exist_ok=True)


In [1]:
def load_history(filename="full_history.json") -> List[Dict]:
    with open(filename, "r", encoding="utf-8") as f:
        return json.load(f)

def view_subsystem_history(history: List[Dict], subsystem: str):
    """
    Выводит все записи для заданной подсистемы (law, politics, science, economics, media).
    """
    entries = [e for e in history if e.get("agent") == subsystem]
    if not entries:
        print(f"Нет записей для подсистемы '{subsystem}'")
        return

    print(f"\n=== Подсистема: {subsystem.upper()} ===\n")
    for i, e in enumerate(entries, 1):
        # Извлекаем год из поля input (если оно начинается с "Год X.")
        year_info = e["input"].split(".")[0] if e["input"] else "?"
        print(f"[{i}] {year_info}")
        print(f"Входные данные:\n{e['input']}\n")
        print(f"Ответ агента:\n{e['output']}\n")
        print("-" * 80)

hist = load_history("history_year_1_20260308-175210.json")
view_subsystem_history(hist, "politics")
#view_subsystem_history(hist, "science")
#view_subsystem_history(hist, "media")

NameError: name 'List' is not defined

In [5]:
hist

[{'agent': 'law',
  'input': 'Год 3. Поступили сообщения:\n- От politics/Правительство: Направляет законопроект об экологических надбавках к налогу на прибыль для компаний, нарушающих международные обязательства по сохранению тропических лесов, с просьбой провести правовую экспертизу\n\nДополнительный контекст: Наука сообщает об ускорении вырубки лесов Амазонии.',
  'output': 'Год 3 завершён.\n\nПодсистема **Право** действовала в рамках своей операциональной замкнутости, реагируя на раздражение от политической подсистемы (запрос на экспертизу законопроекта) исключительно через призму кода **законно/незаконно**. \n\n### Ключевые итоги года:\n- Право **не оценивало** экологическую целесообразность или экономическую эффективность экологических надбавок — это вне его компетенции.\n- Право **не поддерживало и не отвергало** политическую цель законопроекта (сдерживание вырубки Амазонии), но **проверило его соответствие правовым стандартам**: определённость нормы, соблюдение процедур, констит

In [3]:
import json
import glob
import os
from typing import List, Dict

def merge_history_files(pattern: str = "history_year_*.json", output_file: str = "merged_history.json") -> List[Dict]:
    """
    Собирает все файлы, соответствующие pattern, загружает их содержимое (списки записей)
    и объединяет в один список. Сохраняет результат в output_file.
    Возвращает объединённый список.
    """
    all_entries = []
    files = sorted(glob.glob(pattern))  # сортировка по имени файла (предполагаем, что имена содержат дату)
    for fname in files:
        try:
            with open(fname, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    all_entries.extend(data)
                else:
                    print(f"Файл {fname} содержит не список, пропущен.")
        except Exception as e:
            print(f"Ошибка при чтении {fname}: {e}")

    # Дополнительно можно отсортировать записи по времени, если в них есть поле с timestamp
    # Но в текущей структуре такого поля нет, поэтому оставляем как есть.

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(all_entries, f, ensure_ascii=False, indent=2)

    print(f"Объединённая история сохранена в {output_file}. Всего записей: {len(all_entries)}")
    return all_entries

In [5]:
RES_DIR = "Covid_experiments/2"
merged = merge_history_files(RES_DIR+"/"+"history_year_*.json", RES_DIR+"/"+"covid_full_history.json")

Объединённая история сохранена в Covid_experiments/2/covid_full_history.json. Всего записей: 27


In [9]:
def view_subsystem_history(history: List[Dict], subsystem: str, year: int = None):
    """
    Выводит записи для заданной подсистемы. Если указан year, показывает только записи этого года.
    """
    entries = [e for e in history if e.get("agent") == subsystem]
    if year:
        # Извлекаем год из input (ожидается формат "Год X. ...")
        entries = [e for e in entries if e["input"].startswith(f"Год {year}.")]

    if not entries:
        print(f"Нет записей для подсистемы '{subsystem}'" + (f" за {year} год" if year else ""))
        return

    print(f"\n=== Подсистема: {subsystem.upper()} ===\n")
    for i, e in enumerate(entries, 1):
        year_info = e["input"].split(".")[0] if e["input"] else "?"
        print(f"[{i}] {year_info}")
        print(f"Входные данные:\n{e['input']}\n")
        print(f"Ответ агента:\n{e['output']}\n")
        print("-" * 80)

merged = merge_history_files()
view_subsystem_history(merged, "politics", year=3)

Объединённая история сохранена в Covid_experiments/2/merged_history.json. Всего записей: 27

=== Подсистема: POLITICS ===

[1] Год 3
Входные данные:
Год 3. Поступили сообщения:
- От science/Научный совет по инфекционным болезням: Предоставляет предварительную оценку рисков: высокий потенциал передачи, средняя тяжесть, рекомендует усиление эпиднадзора и подготовку к карантинным мерам
- От economics/Центральный банк: Предупреждение: введение карантина без компенсационных выплат приведёт к дефолтам в малом бизнесе и росту безработицы
- От media/Всероссийский телеканал "Новости": Запрос на участие представителя в прямом эфире для разъяснения мер мониторинга и готовности системы здравоохранения

Дополнительный контекст: Медики докладывают, о появлении нового опасного заболевания, 
которое может протекать как в форме острой респираторной вирусной инфекции лёгкого течения, так и в тяжёлой форме. 
К наиболее распространённым симптомам заболевания относятся повышенная температура тела, утомляем